# MQAR investigation -- ridge (kappa*) sweep + conditioning diagnostics

The full run showed `flash_nystrom` (kappa*=1e3) **collapsing on 1/3 seeds** (recall 27%) while the `nystrom_reference` (which ran kappa*=0) was rock-solid at 88.6%. Two things were confounded: the **ridge** (kappa*=1e3 vs 0) and the **kernel** (FN vs pure-torch). This notebook isolates them.

**Hypothesis:** MQAR has seq_len 256, m=64 landmarks -> seg_len=4, so each landmark pools only 4 tokens and K2 is *well-conditioned*. The ridge is therefore unnecessary here, and a too-strong ridge (kappa*=1e3) only adds bias that can strand a borderline seed. Raising kappa* (weaker ridge, toward kappa*=0) should remove the collapse.

**What it does:** sweeps kappa* x backend x seed (all backends honor kappa* now, including the reference, so the comparison is matched), with `--diag` logging cond(K2), cond(M), and the ridged-pinv residual on a fixed probe batch each eval.

**GPU:** sm_80+ (A100/L4). Resumable (skips finished runs).

## 0. GPU check

In [ ]:
import torch
name = torch.cuda.get_device_name(); cap = torch.cuda.get_device_capability()
print(name, '| compute capability', cap)
assert cap[0] >= 8, f'Need sm_80+ (A100/L4); this is {name}. Switch runtime.'

## 1. Clone + build (cwd-safe)

In [ ]:
REPO_URL = 'https://github.com/athrva98/FlashNystrom.git'
REPO_DIR = '/content/flashnystrom'   # every cell cd's here; cwd state can't break paths
%cd /content
!rm -rf flashnystrom
!git clone --recurse-submodules $REPO_URL flashnystrom
%cd {REPO_DIR}
!cd "{REPO_DIR}" && git submodule update --init --recursive
import os, torch
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
os.environ['FLASH_NYSTROM_LAX_BUILD'] = '1'
!cd "{REPO_DIR}" && pip install -e . --no-build-isolation

## 2. Drive + sweep config
Edit `KAPPAS`/`BACKENDS`/`SEEDS` to trade cost vs coverage. Default ~30 runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_NAME = 'mqar_investigation'   # change (e.g. add _j16) for a fresh sweep so results don't mix
OUTDIR = '/content/drive/MyDrive/flashnystrom_runs/' + RUN_NAME
RAW = OUTDIR + '/raw'
os.makedirs(RAW, exist_ok=True)

# Ridge strength: SMALL kappa* = STRONG ridge; kappa*=0 = no ridge (vanilla).
# 1e3 = shipping default (STL); 1e5, 1e7 = progressively weaker ridges toward a no-op.
# HIGH-kappa* / NO-OP CAVEAT: as kappa* -> inf the ridge lambda -> 0, so a ridge far above
# the problem's own conditioning SHOULD be a no-op -- BUT only if Newton-Schulz converges on
# M = K2^T K2 + lambda*I. kappa>0 uses the normal-equations path, whose conditioning
# approaches cond(K2)^2 as lambda->0, so a weak ridge needs MORE iterations. NEWTON_ITER is
# raised to 16 here so the high-kappa* rows test 'is the ridge a no-op', not 'did NS run out
# of iterations'. (The earlier sweep that showed the ridge hurting used J=6.)
KAPPAS      = [0.0, 1000.0, 100000.0, 1.0e7]
BACKENDS    = ['sdpa', 'nystrom_reference', 'flash_nystrom', 'flash_nystrom_tc']
SEEDS       = [0, 1, 2]          # raise to [0,1,2,3,4] for better collapse-rate stats
NEWTON_ITER = 16                 # >6 so the weakly-ridged (high-kappa*) normal matrix can converge
COLLAPSE_THRESHOLD = 50.0        # recall below this = a collapsed run
print('results ->', OUTDIR)

## 3. Run the sweep
`--diag --autobatch`. sdpa ignores kappa* (run once per seed). Each run saves recall + final conditioning to `raw/`.

In [ ]:
# Driving train.py via paper.mqar.runner (shared command construction + output parsing).
import sys, json
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from paper.mqar.runner import run_train, already_done, finite_diag

def run_one(be, ks, s):
    tag = f'{be}_k{ks:g}_j{NEWTON_ITER}_seed{s}'
    out = f'{RAW}/mqar_{tag}.json'
    if already_done(out):
        print('skip', tag); return
    res = run_train(cwd=REPO_DIR, stream=True, log_path=f'{RAW}/mqar_{tag}.log',
                    backend=be, seed=s, kappa_star=ks, seq_len=256, num_kv_pairs=16,
                    num_landmarks=64, newton_iter=NEWTON_ITER, autobatch=True, diag=True)
    rec = res['recall']
    # finite_diag drops the nan cond_K2/cond_M/pinv_resid that sdpa reports
    # (it has no landmarks), keeping them out of the JSON record.
    diag = finite_diag(res)
    json.dump({'backend': be, 'kappa_star': ks, 'seed': s, 'newton_iter': NEWTON_ITER, 'recall': rec,
               'collapsed': (rec is not None and rec < COLLAPSE_THRESHOLD), **diag},
              open(out, 'w'), indent=2)
    print(f'{tag}: recall={rec} {diag}')

for be in BACKENDS:
    kappas = [0.0] if be == 'sdpa' else KAPPAS   # sdpa is kappa-independent
    for ks in kappas:
        for s in SEEDS:
            run_one(be, ks, s)
print('sweep done')

## 4. Aggregate -- collapse rate + recall + conditioning per (backend, kappa*)

In [ ]:
import glob, json, statistics as st
rows = [json.load(open(f)) for f in glob.glob(f'{RAW}/mqar_*_seed*.json')]
groups = {}
for r in rows:
    groups.setdefault((r['backend'], r['kappa_star']), []).append(r)
hdr = ('backend', 'kappa*', 'recall(mean)', 'survivors', 'collapsed', 'cond_K2', 'pinv_resid')
print(f'{hdr[0]:20s} {hdr[1]:>8s} {hdr[2]:>13s} {hdr[3]:>10s} {hdr[4]:>10s} {hdr[5]:>9s} {hdr[6]:>11s}')
summary = {}
for (be, ks), rs in sorted(groups.items()):
    recs = [x['recall'] for x in rs if x['recall'] is not None]
    surv = [x['recall'] for x in rs if not x['collapsed'] and x['recall'] is not None]
    nco = sum(1 for x in rs if x['collapsed'])
    cK2 = [x['cond_K2'] for x in rs if 'cond_K2' in x]
    res = [x['pinv_resid'] for x in rs if 'pinv_resid' in x]
    mean_all = st.mean(recs) if recs else float('nan')
    mean_sv = st.mean(surv) if surv else float('nan')
    ck2 = st.median(cK2) if cK2 else float('nan')
    pr = st.median(res) if res else float('nan')
    print(f'{be:20s} {ks:8.0f} {mean_all:13.2f} {mean_sv:10.2f} {nco:>6d}/{len(rs):<3d} {ck2:9.1e} {pr:11.1e}')
    summary[f'{be}|{ks:g}'] = {'mean_recall': mean_all, 'mean_survivors': mean_sv,
                               'n_collapsed': nco, 'n': len(rs), 'recalls': recs,
                               'median_cond_K2': ck2, 'median_pinv_resid': pr}
json.dump(summary, open(f'{OUTDIR}/mqar_investigation_summary.json', 'w'), indent=2)
print('\nwrote', OUTDIR + '/mqar_investigation_summary.json')
print('\nReading the table:')
print(' - cond_K2 ~ O(10-1e3) => landmark Gram is benign at N=256 (no conditioning need).')
print(' - collapses vanish as kappa* rises (1e3 -> 1e5 -> 0) => the RIDGE caused them, not the kernel.')
print(' - reference also collapses at kappa*=1e3 => confirms ridge, not FN-specific.')

## 5. Conditioning trajectory (one run)
Confirms cond(K2) stays small through training at N=256 (no blow-up).

In [ ]:
import glob, re
logs = sorted(glob.glob(f'{RAW}/mqar_flash_nystrom_k1000_seed*.log'))
if logs:
    txt = open(logs[0]).read()
    print('trajectory from', os.path.basename(logs[0]), '(epoch: recall | cond_K2 | pinv_resid)')
    pat = r'epoch\s+(\d+)/\d+.*?recall ([\d.]+)%.*?cond_K2 ([\d.eE+-]+).*?pinv_resid ([\d.eE+-]+)'
    for m in re.finditer(pat, txt):
        e, rec, ck2, pr = m.groups()
        print(f'  ep{int(e):4d}: recall {float(rec):5.1f}%  cond_K2 {float(ck2):.2e}  pinv_resid {float(pr):.2e}')
else:
    print('no flash_nystrom k1000 logs yet')